# 1. Process PyNNLF Output: Model Comparison on ASHD, ACAP-EDP, SunDance and Fluvius

## Purpose
Creates the shared 1-day comparison tables for the journal article: the test nRMSE table for 14 models on 4 datasets, plus the Ausgrid Solar Home Dataset (ASHD) versus Australian Centre for Advanced Photovoltaics (ACAP), Energy Data Platform (ACAP-EDP) tables that the figures and the runtime table are built from. It does not run PyNNLF; the experiments come from `scripts/run_model_comparison_batch.py`.

## Inputs
- `RECAP_PATHS`: the `a1_experiment_result.csv` recap of each result store (`experiment_result/` for the original ASHD runs, `experiment_result_databricks/` for the runs made on Databricks)

## Run Flow
1. Setup And Paths
2. Load And Validate The 1-Day Results
3. Build And Export Comparison Tables
4. Debug Checklist

## Outputs
- `RESULTS_DIR / "model_comparison_fh8_combined_recap.csv"`: every model on every dataset
- `RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_test_nrmse.csv"`: test nRMSE, 14 models by 4 datasets
- `RESULTS_DIR / "ashd_acap_edp_148hh_fh8_combined_recap.csv"`: the 12 original models on ASHD and ACAP-EDP, read by the figure scripts
- `RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_comparison.csv"`
- `RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_stddev_comparison.csv"`
- `RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_key_models_train_test_runtime.csv"`


## 1. Setup And Paths

Results are read from both result stores. The four datasets and their labels, the twelve original models and the two added deep-learning models are fixed here.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_publication_project(start: Path | None = None) -> Path:
    """Walk up from the working directory to the publication workspace."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


PROJECT_DIR = find_publication_project()
RESULTS_DIR = PROJECT_DIR / "results" / "01_ashd_acap_edp_148hh_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Result stores, in order of precedence when a combination appears in both.
RECAP_PATHS = {
    "publication_journal_article_1_experiment_result": PROJECT_DIR / "experiment_result" / "a1_experiment_result.csv",
    "publication_journal_article_1_experiment_result_databricks": PROJECT_DIR / "experiment_result_databricks" / "a1_experiment_result.csv",
}

FH8_MINUTES = 1440
DATASET_LABELS = {
    "ds20": "ASHD_148hh_weather",
    "ds11": "ACAP-EDP_148hh_weather",
    "ds38": "SunDance_hourly_weather",
    "ds40": "Fluvius_pv_30min_weather",
}
DATASET_NAMES = {
    "ASHD_148hh_weather": "ASHD",
    "ACAP-EDP_148hh_weather": "ACAP-EDP",
    "SunDance_hourly_weather": "SunDance",
    "Fluvius_pv_30min_weather": "Fluvius",
}
ORIGINAL_DATASETS = ["ASHD_148hh_weather", "ACAP-EDP_148hh_weather"]

MODEL_ORDER = ['m1_naive_hp1', 'm2_snaive_hp2', 'm3_ets_hp1', 'm4_arima_hp1', 'm6_lr_hp1', 'm7_ann_hp1', 'm8_dnn_hp1', 'm9_rt_hp3', 'm10_rf_hp1', 'm13_lstm_hp2', 'm16_prophet_hp1', 'm17_xgb_hp1']
ADDED_MODELS = ['m19_nhits_hp2', 'm20_patchtst_hp2']
ALL_MODELS = MODEL_ORDER + ADDED_MODELS
print(f"Results directory: {RESULTS_DIR.relative_to(PROJECT_DIR)}")


## 2. Load And Validate The 1-Day Results

Every recap that exists is read; a combination found in more than one store keeps the row from the first store listed. The table needs every model on every dataset at the 1-day horizon, so a missing combination stops the notebook and names what is missing.

In [ ]:
required = {"dataset_no", "forecast_horizon_min", "model_name", "train_nRMSE", "test_nRMSE", "test_nRMSE_stddev", "runtime_ms"}
frames = []
for source, path in RECAP_PATHS.items():
    if not path.exists():
        print(f"No recap at {path.relative_to(PROJECT_DIR)}; skipping")
        continue
    recap = pd.read_csv(path)
    missing_columns = required - set(recap.columns)
    if missing_columns:
        raise ValueError(f"{path.name} is missing required columns: {sorted(missing_columns)}")
    recap["result_source"] = source
    frames.append(recap)
if not frames:
    raise FileNotFoundError("No recap found. Run scripts/run_model_comparison_batch.py first.")

results = pd.concat(frames, ignore_index=True)
results = results.loc[
    results["dataset_no"].isin(DATASET_LABELS)
    & pd.to_numeric(results["forecast_horizon_min"], errors="coerce").eq(FH8_MINUTES)
    & results["model_name"].isin(ALL_MODELS)
].copy()
results = results.drop_duplicates(subset=["dataset_no", "model_name"], keep="first")
results["forecast_horizon_min"] = FH8_MINUTES
results["dataset_label"] = results["dataset_no"].map(DATASET_LABELS)
results["model_name"] = pd.Categorical(results["model_name"], categories=ALL_MODELS, ordered=True)
results = results.sort_values(["dataset_label", "model_name"])

expected = {(label, model) for label in DATASET_LABELS.values() for model in ALL_MODELS}
found = set(zip(results["dataset_label"], results["model_name"].astype(str)))
missing = sorted(expected - found)
if missing:
    raise ValueError(f"{len(missing)} dataset/model combinations are missing at the 1-day horizon: {missing}")
display(results.pivot_table(index="model_name", columns="dataset_label", values="test_nRMSE", aggfunc="first", observed=False).round(2))


## 3. Build And Export Comparison Tables

The paper's nRMSE table covers all 14 models and 4 datasets. The remaining outputs keep their original scope, the 12 original models on ASHD and ACAP-EDP, because the figures and the runtime table are built from them.

In [ ]:
def wide_metric_by_dataset(frame, metric_column, dataset_labels, models):
    """Pivot one metric to models (rows) by datasets (columns) in a fixed order."""
    table = frame.pivot_table(index="model_name", columns="dataset_label", values=metric_column, aggfunc="first", observed=False)
    table = table.reindex(index=models)[dataset_labels]
    table.index.name = "model_hp"
    table.columns.name = None
    return table


PAPER_MODEL_ORDER = ['m17_xgb_hp1', 'm8_dnn_hp1', 'm6_lr_hp1', 'm10_rf_hp1', 'm7_ann_hp1', 'm1_naive_hp1', 'm3_ets_hp1', 'm9_rt_hp3', 'm13_lstm_hp2', 'm2_snaive_hp2', 'm16_prophet_hp1', 'm4_arima_hp1', *ADDED_MODELS]
PAPER_KEY_MODELS = ['m17_xgb_hp1', 'm1_naive_hp1', 'm4_arima_hp1']
PAPER_MODEL_LABELS = {'m1_naive_hp1': 'naive_hp1', 'm2_snaive_hp2': 'snaive_hp2', 'm3_ets_hp1': 'ets_hp1', 'm4_arima_hp1': 'arima_hp1', 'm6_lr_hp1': 'lr_hp1', 'm7_ann_hp1': 'ann_hp1', 'm8_dnn_hp1': 'dnn_hp1', 'm9_rt_hp3': 'rt_hp3', 'm10_rf_hp1': 'rf_hp1', 'm13_lstm_hp2': 'lstm_hp2', 'm16_prophet_hp1': 'prophet_hp1', 'm17_xgb_hp1': 'xgb_hp1', 'm19_nhits_hp2': 'nhits_hp2', 'm20_patchtst_hp2': 'patchtst_hp2'}


def short_model_name(model_name):
    return PAPER_MODEL_LABELS.get(str(model_name), str(model_name))


common_columns = ["experiment_no", "exp_date", "dataset_no", "dataset_label", "forecast_horizon_min", "model_no", "hyperparameter_no", "model_name", "runtime_ms", "train_RMSE", "train_RMSE_stddev", "test_RMSE", "test_RMSE_stddev", "train_nRMSE", "train_nRMSE_stddev", "test_nRMSE", "test_nRMSE_stddev", "result_source"]
for column in common_columns:
    if column not in results.columns:
        results[column] = pd.NA
all_combined = results[common_columns].copy()

# The 12 original models on the two Australian datasets feed the figures and the runtime table.
combined = all_combined.loc[all_combined["dataset_label"].isin(ORIGINAL_DATASETS) & all_combined["model_name"].astype(str).isin(MODEL_ORDER)].copy()
combined["model_name"] = pd.Categorical(combined["model_name"].astype(str), categories=MODEL_ORDER, ordered=True)
combined = combined.sort_values(["dataset_label", "model_name"])
comparison_nrmse = wide_metric_by_dataset(combined, "test_nRMSE", ORIGINAL_DATASETS, MODEL_ORDER)
comparison_stddev = wide_metric_by_dataset(combined, "test_nRMSE_stddev", ORIGINAL_DATASETS, MODEL_ORDER)


def get_single_value(frame, dataset_label, model_name, column):
    match = frame.loc[frame["dataset_label"].eq(dataset_label) & frame["model_name"].astype(str).eq(model_name), column]
    if match.empty:
        raise ValueError(f"Missing {column} for {dataset_label} / {model_name}")
    return pd.to_numeric(match.iloc[0], errors="coerce")


# The paper table: every model on every dataset, ordered by ASHD accuracy.
paper_table2 = (
    wide_metric_by_dataset(all_combined, "test_nRMSE", list(DATASET_LABELS.values()), ALL_MODELS)
    .reindex(PAPER_MODEL_ORDER)
    .rename(index=short_model_name, columns=DATASET_NAMES)
    .sort_values("ASHD", ascending=True, kind="mergesort")
    .round(1)
)

paper_rows = []
for model in PAPER_KEY_MODELS:
    row = {"Model Name": short_model_name(model)}
    for dataset_label, dataset_name in [("ACAP-EDP_148hh_weather", "ACAP-EDP"), ("ASHD_148hh_weather", "ASHD")]:
        row[f"{dataset_name} Train nRMSE (%)"] = get_single_value(combined, dataset_label, model, "train_nRMSE")
        row[f"{dataset_name} Test nRMSE (%)"] = get_single_value(combined, dataset_label, model, "test_nRMSE")
        row[f"{dataset_name} Training Time (s)"] = get_single_value(combined, dataset_label, model, "runtime_ms") / 1000.0
    paper_rows.append(row)
paper_table3 = pd.DataFrame(paper_rows)
for column in paper_table3.columns.drop("Model Name"):
    paper_table3[column] = pd.to_numeric(paper_table3[column], errors="coerce").round(1)

all_combined.to_csv(RESULTS_DIR / "model_comparison_fh8_combined_recap.csv", index=False)
combined.to_csv(RESULTS_DIR / "ashd_acap_edp_148hh_fh8_combined_recap.csv", index=False)
comparison_nrmse.to_csv(RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_comparison.csv")
comparison_stddev.to_csv(RESULTS_DIR / "ashd_acap_edp_148hh_fh8_nrmse_stddev_comparison.csv")
paper_table2.to_csv(RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_test_nrmse.csv")
paper_table3.to_csv(RESULTS_DIR / "paper_table_ashd_acap_edp_148hh_key_models_train_test_runtime.csv", index=False)

display(comparison_nrmse.round(3))
display(comparison_stddev.round(3))
display(paper_table2)
display(paper_table3)


## 4. Debug Checklist

If this fails, check that the recaps exist, that `scripts/run_model_comparison_batch.py` has finished (its `--dry-run` lists what is still pending), and that every dataset has all 14 models at 1440 minutes.